# Giá cả là đúng

Bước cuối cùng là xây dựng giao diện người dùng.

Chúng ta sẽ sử dụng các khía năng nâng cao hơn của Gradio — xây dựng từng phần một.

### Tóm tắt quy trình của notebook

Notebook này xây dựng giao diện web cuối cùng cho ứng dụng săn deal tự động. Sau khi có cơ chế agent, phần tiếp theo sẽ hiển thị dữ liệu deal dưới dạng bảng và cho phép người dùng tương tác trực tiếp với UI.

### Ý nghĩa chính của notebook

Mục tiêu của notebook là biến quy trình tìm kiếm deal từ logic backend thành một ứng dụng có thể tương tác trên trình duyệt. Dữ liệu được hiển thị và chọn lọc trong giao diện, giúp người dùng quan sát các cơ hội mua hàng và nhận cảnh báo từ agent.


In [ ]:
import gradio as gr
from deal_agent_framework import DealAgentFramework
from agents.deals import Opportunity, Deal

# Cell này import các thư viện và lớp cần thiết để dựng giao diện và thao tác với dữ liệu deal.
# Gradio dùng để tạo UI, còn DealAgentFramework và các model Opportunity/Deal là thành phần backend của hệ thống.


In [ ]:
with gr.Blocks(title="The Price is Right", fill_width=True) as ui:

    with gr.Row():
        gr.Markdown('<div style="text-align: center;font-size:24px">The Price is Right - Deal Hunting Agentic AI</div>')
    with gr.Row():
        gr.Markdown('<div style="text-align: center;font-size:14px">Autonomous agent framework that finds online deals, collaborating with a proprietary fine-tuned LLM deployed on Modal, and a RAG pipeline with a frontier model and Chroma.</div>')
        

ui.launch(inbrowser=True)

# Cell này tạo giao diện đầu tiên cho ứng dụng, chỉ gồm tiêu đề và mô tả tổng quan.
# `gr.Blocks` là khung chính của Gradio, dùng để gom các thành phần UI lại với nhau.
# `gr.Row()` và `gr.Markdown()` giúp tạo bố cục và hiển thị văn bản trên trang.
# Dòng cuối `ui.launch(inbrowser=True)` mở giao diện ngay trong trình duyệt để xem hiệu quả.


In [ ]:
# Updated to change from height to max_height due to change in Gradio v5
# With much thanks to student Ed B. for raising this

with gr.Blocks(title="The Price is Right", fill_width=True) as ui:

    initial_deal = Deal(product_description="Example description", price=100.0, url="https://cnn.com")
    initial_opportunity = Opportunity(deal=initial_deal, estimate=200.0, discount=100.0)
    opportunities = gr.State([initial_opportunity])

    def get_table(opps):
        return [[opp.deal.product_description, opp.deal.price, opp.estimate, opp.discount, opp.deal.url] for opp in opps]

    with gr.Row():
        gr.Markdown('<div style="text-align: center;font-size:24px">"The Price is Right" - Deal Hunting Agentic AI</div>')
    with gr.Row():
        gr.Markdown('<div style="text-align: center;font-size:14px">Deals surfaced so far:</div>')
    with gr.Row():
        opportunities_dataframe = gr.Dataframe(
            headers=["Description", "Price", "Estimate", "Discount", "URL"],
            wrap=True,
            column_widths=[4, 1, 1, 1, 2],
            row_count=10,
            col_count=5,
            max_height=400,
        )

    ui.load(get_table, inputs=[opportunities], outputs=[opportunities_dataframe])

ui.launch(inbrowser=True)

# Cell này bổ sung dữ liệu ban đầu và hiển thị danh sách deal dưới dạng bảng.
# `initial_deal` và `initial_opportunity` tạo một mẫu dữ liệu để UI có gì đó hiển thị ngay khi mở.
# `gr.State` lưu trạng thái dữ liệu trong ứng dụng, giúp dữ liệu có thể được cập nhật theo thời gian.
# `get_table()` biến danh sách opportunity thành dạng dữ liệu phù hợp để Dataframe hiển thị.
# `gr.Dataframe` tạo bảng với các cột: mô tả, giá, ước tính, giảm giá, URL.
# `ui.load(...)` gọi hàm `get_table` khi UI khởi động, đảm bảo bảng được render ngay từ đầu.


In [ ]:
agent_framework = DealAgentFramework()
agent_framework.init_agents_as_needed()

with gr.Blocks(title="The Price is Right", fill_width=True) as ui:

    initial_deal = Deal(product_description="Example description", price=100.0, url="https://cnn.com")
    initial_opportunity = Opportunity(deal=initial_deal, estimate=200.0, discount=100.0)
    opportunities = gr.State([initial_opportunity])

    def get_table(opps):
        return [[opp.deal.product_description, opp.deal.price, opp.estimate, opp.discount, opp.deal.url] for opp in opps]

    def do_select(opportunities, selected_index: gr.SelectData):
        row = selected_index.index[0]
        opportunity = opportunities[row]
        agent_framework.planner.messenger.alert(opportunity)

    with gr.Row():
        gr.Markdown('<div style="text-align: center;font-size:24px">"The Price is Right" - Deal Hunting Agentic AI</div>')
    with gr.Row():
        gr.Markdown('<div style="text-align: center;font-size:14px">Deals surfaced so far:</div>')
    with gr.Row():
        opportunities_dataframe = gr.Dataframe(
            headers=["Description", "Price", "Estimate", "Discount", "URL"],
            wrap=True,
            column_widths=[4, 1, 1, 1, 2],
            row_count=10,
            col_count=5,
            max_height=400,
        )

    ui.load(get_table, inputs=[opportunities], outputs=[opportunities_dataframe])
    opportunities_dataframe.select(do_select, inputs=[opportunities], outputs=[])

ui.launch(inbrowser=True)

# Cell này đưa tính tương tác vào UI: khi người dùng click vào một dòng trong bảng, hệ thống sẽ gửi deal đó tới planner agent.
# `DealAgentFramework()` khởi tạo nền tảng agent, cho phép các agent tương tác với nhau khi có sự kiện xảy ra.
# `do_select` đọc dòng được chọn trong Dataframe bằng `selected_index.index[0]`, lấy opportunity tương ứng, rồi gọi alert() để cảnh báo agent.
# Đây là bước quan trọng để biến UI từ “hiển thị” thành “xử lý sự kiện” và tạo trải nghiệm agentic.


# Thời gian cho code

Bây giờ chúng ta chuyển sang file `price_is_right.py` để xem phần logic hoàn chỉnh.

### Tóm tắt quy trình của notebook

Từ đây, notebook không còn chỉ demo UI nữa mà chuyển sang phần triển khai code chính của ứng dụng. Mục tiêu là thực thi logic tìm deal thực tế trong file Python phụ trách phần agentic workflow.

### Ý nghĩa chính của notebook

Cell này là cầu nối giữa giao diện minh họa và ứng dụng thực tế. Nó nhấn mạnh rằng các UI ở trên chỉ là lớp phủ tương tác, trong khi logic điều phối đã được đưa vào tệp Python riêng.


In [ ]:
# Reset bộ nhớ về 2 deal đã được tìm thấy trong quá khứ

from deal_agent_framework import DealAgentFramework
DealAgentFramework.reset_memory()

# Cell này đặt lại trạng thái bộ nhớ của hệ thống để quá trình chạy lại bắt đầu từ cùng một điểm chuẩn.
# Việc reset memory giúp tránh việc dữ liệu cũ ảnh hưởng đến lần chạy mới, đặc biệt khi bạn đang test hoặc demo lại ứng dụng.


In [ ]:
import logging
root = logging.getLogger()
root.setLevel(logging.INFO)

# Cell này cấu hình logging ở mức INFO để hiển thị thông tin quan trọng trong quá trình thực thi agent.
# Khi app chạy, bạn sẽ thấy log về hoạt động tìm kiếm deal, phản hồi agent và các bước xử lý nền.


# Chạy sản phẩm cuối cùng

## Chỉ cần nhấn Shift + Enter trong cell tiếp theo, và để các deal chảy vào thôi!!

### Tóm tắt quy trình của notebook

Đây là bước chạy ứng dụng cuối cùng sau khi tất cả thành phần UI và logic agent đã được kết nối. Khi chạy cell này, hệ thống sẽ bắt đầu quy trình săn deal tự động và hiển thị kết quả trên giao diện.

### Ý nghĩa chính của notebook

Cell này thể hiện mục tiêu cuối của toàn bộ dự án: cho ứng dụng hoạt động thật và tự động phát hiện cơ hội giảm giá từ các nguồn dữ liệu online.


In [ ]:
!uv run price_is_right.py

# Cell này chạy file chính `price_is_right.py` bằng công cụ `uv`, tức là khởi động ứng dụng deal hunting end-to-end.
# Khi chạy xong, ứng dụng sẽ thực hiện các bước quy hoạch, tìm kiếm deal, đánh giá cơ hội và hiển thị trong giao diện.


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#a22;">Nhưng chờ đã!! Còn nhiều hơn thế nữa..</h2>
            <span style="color:#a22;">Nếu bạn vẫn chưa chán giá sản phẩm thì 😂 tôi đã mở rộng nó thêm nữa!<br/>
            Nếu bạn xem trong repo của tôi <a href="https://github.com/ed-donner/tech2ai">tech2ai</a>, ở segment4 có một phiên bản dùng OpenAI Agents SDK cho AutonomousPlanningAgent và có thêm MCP servers. Nếu bạn thấy hứng thú với Agents và MCP, và muốn học sâu hơn, tôi cũng có một khóa học đi kèm. Cùng với 2 khóa học đi kèm khác, mọi thứ đều được sắp xếp trong <a href="https://edwarddonner.com/2025/05/28/connecting-my-courses-become-an-llm-expert-and-leader/">chương trình AI toàn diện</a>.
            </span>
        </td>
    </tr>
</table>

### Ý nghĩa chính của notebook

Cell này là lời nhắc rằng dự án này có thể tiếp tục mở rộng với các công nghệ mới hơn như OpenAI Agents SDK, MCP và hệ thống agent chuyên sâu hơn.


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">CHÚC MỪNG VÀ CẢM ƠN!!!</h2>
            <span style="color:#090;">
                Thật tuyệt khi bạn đã đến cuối cùng! Chúc mừng bạn. Hãy giữ liên lạc với tôi nhé! Tôi có trên <a href="https://www.linkedin.com/in/eddonner/">LinkedIn</a> nếu chúng ta chưa kết nối, và tôi cũng có trên X ở <a href="https://x.com/edwarddonner">@edwarddonner</a>. Và người biên tập của tôi sẽ khó chịu nếu tôi không nhắc thêm một lần nữa: việc học viên đánh giá khóa học trên Udemy tạo ra sự khác biệt rất lớn - đó là một trong những cách chính mà Udemy quyết định có nên hiển thị khóa học cho người khác hay không. <br/><br/>Một lần nữa cảm ơn vì đã chịu đựng tôi trong 8 tuần và đi đến cell cuối cùng! Tôi rất mong được nghe về sự nghiệp AI Engineer của bạn. Nếu bạn đăng trên LinkedIn về việc hoàn thành khóa học và gắn thẻ tôi, tôi sẽ hỗ trợ để khuếch đại thành tựu của bạn. <br/><b>Bạn không thể chọn đúng thời điểm nào hơn để bước vào lĩnh vực này.</b>
            </span>
        </td>
    </tr>
</table>

### Mục tiêu cuối cùng

Sau khi chạy và hiểu toàn bộ notebook, người học không chỉ thấy được cách xây dựng ứng dụng săn deal mà còn nắm được quy trình hoàn chỉnh từ UI, agent framework đến deployment và mở rộng hệ thống ở mức cao hơn.
